# 03 — FASE 2: Processamento e agregações

> **A pergunta desta fase:** como transformo dado *fiel à fonte* em dado que
> *responde perguntas*?

O caminho tem dois saltos bem diferentes:

```
bronze  ──(tipar, limpar, modelar)──►  silver  ──(agregar por pergunta)──►  gold
 texto                                 tipado                              pronto
 fiel à fonte                          modelo estrela                      para consumo
```

E cada salto tem uma regra que não se quebra:

* **silver** — aqui e **só** aqui moram as regras de negócio. "O que é um
  abandono?" tem uma única resposta no projeto inteiro.
* **gold** — aqui não se decide nada, só se soma. Quem consome a gold **não faz
  JOIN e não aplica regra**: só desenha.

In [ ]:
# --- Preparação do ambiente (rode esta célula primeiro) --------------------
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

print("Raiz do projeto:", RAIZ)
print("Python:", sys.version.split()[0])


In [2]:
from f1_pipeline import config
from f1_pipeline.processing.bronze import build_bronze

bronze = build_bronze(config.SEASONS)
bronze["results"].head(3)


22:06:01 | INFO    | f1_pipeline.processing.bronze | BRONZE races                  |     90 linhas | 12 colunas -> races.parquet
22:06:01 | INFO    | f1_pipeline.processing.bronze | BRONZE results                |   1799 linhas | 26 colunas -> results.parquet
22:06:01 | INFO    | f1_pipeline.processing.bronze | BRONZE qualifying             |   1798 linhas | 11 colunas -> qualifying.parquet
22:06:02 | INFO    | f1_pipeline.processing.bronze | BRONZE pit_stops              |   3340 linhas |  9 colunas -> pit_stops.parquet
22:06:02 | INFO    | f1_pipeline.processing.bronze | BRONZE driver_standings       |     89 linhas | 13 colunas -> driver_standings.parquet
22:06:02 | INFO    | f1_pipeline.processing.bronze | BRONZE constructor_standings  |     40 linhas |  9 colunas -> constructor_standings.parquet


,season,round,race_name,date,driver_id,driver_code,driver_number,given_name,family_name,driver_nationality,date_of_birth,constructor_id,constructor_name,constructor_nationality,grid,position,position_text,points,laps,status,time_millis,time_text,fastest_lap_rank,fastest_lap_number,fastest_lap_time,fastest_lap_speed_kph
0,2021,1,Bahrain Grand Prix,2021-03-28,hamilton,HAM,44,Lewis,Hamilton,British,1985-01-07,mercedes,Mercedes,German,2,1,1,25,56,Finished,5523897,1:32:03.897,4,44,1:34.015,207.235
1,2021,1,Bahrain Grand Prix,2021-03-28,max_verstappen,VER,33,Max,Verstappen,Dutch,1997-09-30,red_bull,Red Bull,Austrian,1,2,2,18,56,Finished,5524642,+0.745,2,41,1:33.228,208.984
2,2021,1,Bahrain Grand Prix,2021-03-28,bottas,BOT,77,Valtteri,Bottas,Finnish,1989-08-28,mercedes,Mercedes,German,3,3,3,16,56,Finished,5561280,+37.383,1,56,1:32.090,211.566


## 1. O primeiro salto: tipagem

Parece burocracia. Não é — é o que torna o dado *calculável*.

Três armadilhas que a tipagem resolve, e que aparecem em qualquer projeto:

In [3]:
# ARMADILHA 1: ordenação de texto
tempos_texto = ["1:02.000", "59.000", "1:32.608"]
print("Ordenado como TEXTO :", sorted(tempos_texto))
print("   -> '1:02.000' aparece antes de '59.000': errado, 62s > 59s\n")

from f1_pipeline.processing.silver import tempo_para_segundos

tempos_numero = sorted(tempos_texto, key=tempo_para_segundos)
print("Ordenado como NÚMERO:", tempos_numero)
print("   -> agora sim.")


Ordenado como TEXTO : ['1:02.000', '1:32.608', '59.000']
   -> '1:02.000' aparece antes de '59.000': errado, 62s > 59s

Ordenado como NÚMERO: ['59.000', '1:02.000', '1:32.608']
   -> agora sim.


In [4]:
# ARMADILHA 2: soma de texto concatena em vez de somar
pontos = bronze["results"]["points"].head(3)
print("Valores        :", list(pontos))
print("Soma como texto:", pontos.sum())          # concatena!
print("Soma numérica  :", pd.to_numeric(pontos).sum())


Valores        : ['25', '18', '16']
Soma como texto: 251816
Soma numérica  : 59


In [5]:
# ARMADILHA 3: ausência de valor precisa de um tipo que a represente
serie = pd.Series(["1", "2", None, "4"])
print("to_numeric -> float64 (o None vira NaN, e 1 vira 1.0):")
print(pd.to_numeric(serie, errors="coerce").tolist())
print("\nInt64 (inteiro anulável) mantém inteiro E aceita ausência:")
print(pd.to_numeric(serie, errors="coerce").astype("Int64").tolist())


to_numeric -> float64 (o None vira NaN, e 1 vira 1.0):
[1.0, 2.0, nan, 4.0]

Int64 (inteiro anulável) mantém inteiro E aceita ausência:
[1, 2, <NA>, 4]


## 2. O segundo salto: regras de negócio

Aqui está a parte que exige conhecer o **domínio**, não só o pandas.

### "Abandonou" não é o que parece

A API traz um campo `status` com valores como `Finished`, `Lapped`, `Retired`,
`Did not start`, `Disqualified`. Um piloto com status `Lapped` **terminou a
corrida** — apenas tomou uma volta do líder. Contar `Lapped` como abandono
inflaria os abandonos de ~1 para ~10 por GP.

Pior: o valor mudou de nome. A Ergast original escrevia `"+1 Lap"`; a Jolpica
escreve `"Lapped"`. Quem tivesse fixado o texto antigo no código estaria hoje
com o número errado **sem nenhum erro de execução** — exatamente o tipo de falha
que o notebook 02 existe para pegar.

In [6]:
bronze["results"]["status"].value_counts().head(10)


status
Finished            1087
+1 Lap               210
Lapped               208
Retired              105
+2 Laps               34
Collision             30
Collision damage      17
Accident              17
Engine                 9
Gearbox                9
Name: count, dtype: int64

In [7]:
# Classificado (recebeu posição oficial) != completou a prova.
# Quem roda faltando poucas voltas ainda é classificado.
comparacao = bronze["results"].assign(
    posicao_numerica=lambda d: d["position_text"].str.fullmatch(r"\d+")
)
pd.crosstab(comparacao["status"], comparacao["posicao_numerica"], rownames=["status"],
            colnames=["position_text é numérico?"])


position_text é numérico?,False,True
status,,
+1 Lap,0,210
+2 Laps,0,34
+3 Laps,0,6
+6 Laps,0,1
Accident,16,1
Brakes,3,0
Collision,28,2
Collision damage,14,3
Cooling system,1,0


### Grid 0: quando o dado usa um código no lugar de um valor

`grid = 0` não significa "largou na posição zero": significa **largou do pit
lane**. Se calcularmos "posições ganhas = grid − chegada" com o zero literal, o
piloto que largou do pit e chegou em 15º apareceria tendo *perdido* 15 posições —
quando na verdade ele ganhou várias.

A regra do projeto: `grid = 0` vira "último lugar daquela corrida".

In [8]:
from f1_pipeline.processing.silver import build_fact_result

fato_resultado = build_fact_result(bronze["results"])

pit_lane = fato_resultado[fato_resultado["grid"] == 0]
print(f"{len(pit_lane)} largadas do pit lane nas temporadas carregadas\n")
pit_lane[["season", "rodada", "driver_id", "grid", "grid_efetivo", "position", "posicoes_ganhas"]].head()


27 largadas do pit lane nas temporadas carregadas



,season,rodada,driver_id,grid,grid_efetivo,position,posicoes_ganhas
4,2021,1,perez,0,20,5,15
34,2021,2,vettel,0,20,15,5
132,2021,7,tsunoda,0,20,13,7
195,2021,10,perez,0,20,16,4
212,2021,11,giovinazzi,0,20,13,7


## 3. Modelagem dimensional: dimensões e fatos

O bronze repete o nome do piloto, a nacionalidade e a data de nascimento em
**todas as 1.799 linhas** de resultado. Isso é desperdício e, pior, é fonte de
inconsistência: basta um registro escrever "Verstappen" diferente para o
`groupby` criar dois pilotos.

A solução clássica é o **modelo estrela**:

* **Dimensões** (`dim_`) — *quem, onde, quando*. Uma linha por entidade.
  `dim_driver`, `dim_constructor`, `dim_circuit`, `dim_race`.
* **Fatos** (`fact_`) — *o que aconteceu*. Uma linha por evento, com métricas e
  as chaves das dimensões. `fact_result`, `fact_qualifying`, `fact_pit_stop`.

```
        dim_driver ─┐
     dim_constructor┼─► fact_result ◄── dim_race ──► dim_circuit
                    ┘   (grão: 1 piloto x 1 corrida)
```

O conceito mais importante aqui é o **grão** (*grain*): "o que representa uma
linha desta tabela?". Em `fact_result` o grão é **um piloto em uma corrida**.
Definir o grão antes de escrever qualquer agregação evita 90% dos erros de
contagem dupla.

In [9]:
from f1_pipeline.processing.silver import build_silver

silver = build_silver(bronze)
print()
for nome, df in silver.items():
    tipo = "DIMENSÃO" if nome.startswith("dim") else "FATO    "
    print(f"{tipo} {nome:<26} {len(df):>6} linhas x {df.shape[1]:>2} colunas")


22:06:59 | INFO    | f1_pipeline.processing.silver | SILVER dim_driver                 |     32 linhas |  7 colunas -> dim_driver.parquet
22:06:59 | INFO    | f1_pipeline.processing.silver | SILVER dim_constructor            |     12 linhas |  3 colunas -> dim_constructor.parquet
22:06:59 | INFO    | f1_pipeline.processing.silver | SILVER dim_circuit                |     28 linhas |  6 colunas -> dim_circuit.parquet
22:06:59 | INFO    | f1_pipeline.processing.silver | SILVER dim_race                   |     90 linhas |  6 colunas -> dim_race.parquet
22:06:59 | INFO    | f1_pipeline.processing.silver | SILVER fact_result                |   1799 linhas | 23 colunas -> fact_result.parquet
22:06:59 | INFO    | f1_pipeline.processing.silver | SILVER fact_qualifying            |   1798 linhas | 12 colunas -> fact_qualifying.parquet
22:06:59 | INFO    | f1_pipeline.processing.silver | SILVER fact_pit_stop              |   3340 linhas |  8 colunas -> fact_pit_stop.parquet
22:06:59 | INFO    | 

In [10]:
silver["dim_driver"].head(4)


,driver_id,piloto,sigla,nome,sobrenome,nacionalidade,data_nascimento
0,hamilton,Lewis Hamilton,HAM,Lewis,Hamilton,British,1985-01-07
1,max_verstappen,Max Verstappen,VER,Max,Verstappen,Dutch,1997-09-30
2,bottas,Valtteri Bottas,BOT,Valtteri,Bottas,Finnish,1989-08-28
3,norris,Lando Norris,NOR,Lando,Norris,British,1999-11-13


In [11]:
silver["fact_result"].head(4)


,race_key,season,rodada,driver_id,constructor_id,grid,position,position_text,points,laps,status,tempo_total_ms,volta_rapida_kph,volta_rapida_s,finalizou,abandonou,classificado,motivo_abandono,vitoria,podio,pontuou,grid_efetivo,posicoes_ganhas
0,2021-01,2021,1,hamilton,mercedes,2,1,1,25.0,56,Finished,5523897,207.235,94.015,True,False,True,NaN,True,True,True,2,1
1,2021-01,2021,1,max_verstappen,red_bull,1,2,2,18.0,56,Finished,5524642,208.984,93.228,True,False,True,NaN,False,True,True,1,-1
2,2021-01,2021,1,bottas,mercedes,3,3,3,16.0,56,Finished,5561280,211.566,92.090,True,False,True,NaN,False,True,True,3,0
3,2021-01,2021,1,norris,mclaren,7,4,4,12.0,56,Finished,5570363,206.398,94.396,True,False,True,NaN,False,False,True,7,3


In [12]:
# A tipagem foi aplicada: agora existem números, datas e booleanos de verdade.
silver["fact_result"].dtypes.to_frame("tipo").T


,race_key,season,rodada,driver_id,constructor_id,grid,position,position_text,points,laps,status,tempo_total_ms,volta_rapida_kph,volta_rapida_s,finalizou,abandonou,classificado,motivo_abandono,vitoria,podio,pontuou,grid_efetivo,posicoes_ganhas
tipo,object,Int64,Int64,object,object,Int64,Int64,object,float64,Int64,object,Int64,float64,float64,bool,bool,bool,object,boolean,boolean,bool,Int64,Int64


### Colunas que a fonte não tinha

A silver não só limpa: ela **enriquece**. Estas colunas não existem na API e são
o que torna a análise possível:

| Coluna | Como é calculada | Para que serve |
|---|---|---|
| `finalizou` / `abandonou` | a partir do `status`, com a regra do `Lapped` | confiabilidade |
| `classificado` | `position_text` é numérico? | diferente de "completou" |
| `motivo_abandono` | o `status`, só quando abandonou | análise de falhas |
| `grid_efetivo` | grid, com 0 → último | base do cálculo abaixo |
| `posicoes_ganhas` | `grid_efetivo − position` | desempenho na corrida |
| `vitoria` / `podio` / `pontuou` | faixas de `position` e `points` | contagens do gold |
| `volta_rapida_s` | `"1:32.608"` → `92.608` | comparação de ritmo |

In [13]:
# Um exemplo do enriquecimento: os 5 pilotos que mais ganharam posições
top_ultrapassadores = (
    silver["fact_result"]
    .merge(silver["dim_driver"][["driver_id", "piloto"]], on="driver_id")
    .groupby("piloto", as_index=False)["posicoes_ganhas"]
    .sum()
    .nlargest(5, "posicoes_ganhas")
)
top_ultrapassadores


,piloto,posicoes_ganhas
13,Lance Stroll,112
9,Guanyu Zhou,84
29,Sergio Pérez,67
5,Esteban Ocon,64
15,Lewis Hamilton,46


### Outliers: o pit stop de 9 minutos

Nem todo valor estranho é erro de dado. No GP do Japão de 2024 houve bandeira
vermelha, e o cronômetro da parada continuou correndo: **543 segundos**.

O valor é *verdadeiro*. Mas usá-lo em uma média de "quão rápido é o pit desta
equipe?" responde a pergunta errada. A silver marca esses casos em vez de
apagá-los — quem consome decide se inclui.

In [14]:
paradas = silver["fact_pit_stop"]
atipicas = paradas[paradas["parada_atipica"]]

print(f"{len(atipicas)} paradas atípicas em {len(paradas)} ({100*len(atipicas)/len(paradas):.1f}%)\n")
print(f"Média COM as atípicas  : {paradas['duracao_s'].mean():>7.2f} s")
print(f"Média SEM as atípicas  : {paradas[~paradas['parada_atipica']]['duracao_s'].mean():>7.2f} s")
print(f"Mediana (robusta)      : {paradas['duracao_s'].median():>7.2f} s")
print("\nA maior parada registrada:")
atipicas.nlargest(3, "duracao_s")[["season", "rodada", "driver_id", "volta", "duracao_s"]]


322 paradas atípicas em 3340 (9.6%)

Média COM as atípicas  :  163.46 s
Média SEM as atípicas  :   24.78 s
Mediana (robusta)      :   23.99 s

A maior parada registrada:


,season,rodada,driver_id,volta,duracao_s
1125,2022,10,stroll,1,3069.017
1124,2022,10,vettel,1,3067.301
1126,2022,10,mick_schumacher,1,3065.174


> **Lição.** Média é frágil a outlier; mediana não é. Antes de escolher a
> métrica, pergunte: *qual pergunta eu quero responder?* "Quão rápido é o pit
> desta equipe?" pede a média das paradas normais. "Quanto tempo os carros
> ficaram parados?" pede a soma de todas.

## 4. Portão de qualidade: validando a silver

Agora que existem tipos, dá para validar **semântica** — faixas e regras de
negócio que seriam impossíveis de expressar sobre texto.

In [15]:
from f1_pipeline.quality import get_context, results_to_dataframe, suites, validate_dataframe

contexto = get_context()

resultados_silver = [
    validate_dataframe(silver[nome], fabrica(), asset_name=f"silver_{nome}",
                       context=contexto, build_docs=False)
    for nome, fabrica in suites.SILVER_SUITES.items()
]
results_to_dataframe(resultados_silver)


22:07:38 | INFO    | f1_pipeline.quality.context  | Contexto GX pronto em C:\dev\Estudo-MDS\gx
22:07:38 | INFO    | f1_pipeline.quality.context  | Validando 'silver_fact_result' (1799 linhas) com a suite 'silver_fact_result'
22:07:38 | INFO    | f1_pipeline.quality.context  | APROVADO | silver_fact_result: 7/7 regras aprovadas
22:07:38 | INFO    | f1_pipeline.quality.context  | Validando 'silver_fact_qualifying' (1798 linhas) com a suite 'silver_fact_qualifying'
22:07:38 | INFO    | f1_pipeline.quality.context  | APROVADO | silver_fact_qualifying: 3/3 regras aprovadas


,tabela,regras,aprovadas,reprovadas,taxa_sucesso_%,linhas,status
0,silver_fact_result,7,7,0,100.0,1799,APROVADO
1,silver_fact_qualifying,3,3,0,100.0,1798,APROVADO


## 5. GOLD: uma tabela por pergunta

A camada gold inverte a lógica: em vez de modelar o *dado*, modelamos a
*pergunta*.

| Tabela | Pergunta que responde | Grão |
|---|---|---|
| `agg_driver_season` | Como foi a temporada de cada piloto? | piloto × temporada |
| `agg_constructor_season` | Como foi a temporada de cada equipe? | equipe × temporada |
| `agg_race_summary` | O que aconteceu em cada GP? | corrida |
| `agg_championship_progression` | Como o campeonato evoluiu? | piloto × rodada |
| `agg_circuit_stats` | Que tipo de corrida cada circuito produz? | circuito |
| `agg_teammate_battle` | Quem venceu o duelo interno? | piloto × equipe × temporada |

Repare que **toda** tabela gold já traz os nomes legíveis resolvidos. O app não
faz um único JOIN.

In [16]:
from f1_pipeline.processing.gold import build_gold

gold = build_gold(silver)
print()
for nome, df in gold.items():
    print(f"{nome:<32} {len(df):>5} linhas x {df.shape[1]:>2} colunas")


22:07:48 | INFO    | f1_pipeline.processing.gold  | GOLD agg_driver_season                |     89 linhas | 20 colunas -> agg_driver_season.parquet
22:07:48 | INFO    | f1_pipeline.processing.gold  | GOLD agg_constructor_season           |     40 linhas | 14 colunas -> agg_constructor_season.parquet
22:07:48 | INFO    | f1_pipeline.processing.gold  | GOLD agg_race_summary                 |     90 linhas | 19 colunas -> agg_race_summary.parquet
22:07:48 | INFO    | f1_pipeline.processing.gold  | GOLD agg_championship_progression     |   2006 linhas |  9 colunas -> agg_championship_progression.parquet
22:07:48 | INFO    | f1_pipeline.processing.gold  | GOLD agg_circuit_stats                |     28 linhas | 12 colunas -> agg_circuit_stats.parquet
22:07:49 | INFO    | f1_pipeline.processing.gold  | GOLD agg_teammate_battle              |     90 linhas | 11 colunas -> agg_teammate_battle.parquet

agg_driver_season                   89 linhas x 20 colunas
agg_constructor_season             

In [17]:
gold["agg_driver_season"].query("season == 2024").head(8)[
    ["piloto", "equipe_principal", "corridas", "pontos", "vitorias", "podios",
     "poles", "media_grid", "media_chegada", "taxa_abandono_%"]
]


,piloto,equipe_principal,corridas,pontos,vitorias,podios,poles,media_grid,media_chegada,taxa_abandono_%
65,Max Verstappen,Red Bull,24,399.0,9,14,10,3.54,3.62,4.2
66,Lando Norris,McLaren,24,344.0,4,13,8,3.38,4.29,0.0
67,Charles Leclerc,Ferrari,24,327.0,3,13,2,5.42,4.54,4.2
68,Oscar Piastri,McLaren,24,265.0,2,8,0,5.42,5.12,0.0
69,Carlos Sainz,Ferrari,23,262.0,2,9,1,5.7,5.7,13.0
70,George Russell,Mercedes,24,226.0,2,4,3,5.62,6.75,12.5
71,Lewis Hamilton,Mercedes,24,207.0,2,5,0,8.83,6.96,8.3
72,Sergio Pérez,Red Bull,24,138.0,0,4,0,9.38,9.62,20.8


### Uma agregação que exige cuidado: a evolução do campeonato

Somar pontos por rodada e acumular parece trivial. Tem uma armadilha:
**um piloto que falta a um GP não tem linha naquela rodada**. Se acumularmos
direto, a curva dele desaparece do gráfico naquele ponto.

A solução é *completar a malha*: gerar todas as combinações rodada × piloto da
temporada, preencher com zero quem não correu, e só então acumular.

In [18]:
progressao = gold["agg_championship_progression"].query("season == 2024")

lider_por_rodada = (
    progressao[progressao["posicao_campeonato"] == 1]
    .groupby("rodada", as_index=False)
    .first()[["rodada", "piloto", "pontos_acumulados"]]
)
print("Quem liderou o campeonato de 2024 após cada rodada:")
lider_por_rodada.head(10)


Quem liderou o campeonato de 2024 após cada rodada:


,rodada,piloto,pontos_acumulados
0,1,Max Verstappen,26.0
1,2,Max Verstappen,51.0
2,3,Max Verstappen,51.0
3,4,Max Verstappen,77.0
4,5,Max Verstappen,102.0
5,6,Max Verstappen,120.0
6,7,Max Verstappen,145.0
7,8,Max Verstappen,153.0
8,9,Max Verstappen,178.0
9,10,Max Verstappen,203.0


In [19]:
# Malha completa: nº de pilotos x nº de rodadas, sem buracos
print("Linhas esperadas:", progressao["rodada"].nunique() * progressao["driver_id"].nunique())
print("Linhas na tabela:", len(progressao))


Linhas esperadas: 576
Linhas na tabela: 576


### Uma agregação que exige regra: o duelo entre companheiros

"Quem bate o companheiro de equipe?" é a comparação mais justa da F1: mesmo
carro, mesma estratégia. Mas o cálculo precisa de uma decisão explícita:

**Se um piloto abandona por quebra de motor, o companheiro "venceu" o duelo?**

Nossa resposta: **não**. Um duelo de corrida só conta quando **os dois
completaram a prova** — senão estaríamos medindo confiabilidade do carro, não
desempenho do piloto. Na classificação, os dois sempre participam, então todo GP
conta.

Essa decisão está escrita no código e documentada. Outra equipe poderia decidir
diferente — desde que **decida**, e não deixe o acaso do `groupby` decidir.

In [20]:
duelos_2024 = gold["agg_teammate_battle"].query("season == 2024")
duelos_2024[duelos_2024["duelos_quali"] >= 15].sort_values(
    "aproveitamento_quali_%", ascending=False
)[["equipe", "piloto", "duelos_quali", "vitorias_quali", "aproveitamento_quali_%",
   "duelos_corrida", "vitorias_corrida", "aproveitamento_corrida_%"]]


,equipe,piloto,duelos_quali,vitorias_quali,aproveitamento_quali_%,duelos_corrida,vitorias_corrida,aproveitamento_corrida_%
83,Red Bull,Max Verstappen,24,23,95.8,18,18,100.0
87,Williams,Alexander Albon,23,21,91.3,13,12,92.3
86,Sauber,Valtteri Bottas,24,21,87.5,20,12,60.0
76,McLaren,Lando Norris,24,20,83.3,24,16,66.7
78,Mercedes,George Russell,24,19,79.2,20,13,65.0
68,Aston Martin,Fernando Alonso,24,19,79.2,19,14,73.7
82,RB F1 Team,Yuki Tsunoda,24,18,75.0,18,11,61.1
74,Haas F1 Team,Nico Hülkenberg,24,16,66.7,20,15,75.0
71,Ferrari,Charles Leclerc,24,15,62.5,21,13,61.9
65,Alpine F1 Team,Esteban Ocon,23,12,52.2,17,9,52.9


## 6. Portão de qualidade: reconciliação da gold

O portão da gold pergunta uma coisa específica: **a agregação continua fiel ao
detalhe?** Somar é fácil; somar errado também.

A regra mais elegante aqui é uma relação lógica: **pódios ≥ vitórias**, sempre.
Se um dia essa regra falhar, o bug está na lógica de agregação — não no dado.

In [21]:
resultados_gold = [
    validate_dataframe(gold[nome], fabrica(), asset_name=f"gold_{nome}",
                       context=contexto, build_docs=False)
    for nome, fabrica in suites.GOLD_SUITES.items()
]
results_to_dataframe(resultados_gold)


22:08:30 | INFO    | f1_pipeline.quality.context  | Validando 'gold_agg_driver_season' (89 linhas) com a suite 'gold_agg_driver_season'
22:08:30 | INFO    | f1_pipeline.quality.context  | APROVADO | gold_agg_driver_season: 7/7 regras aprovadas
22:08:30 | INFO    | f1_pipeline.quality.context  | Validando 'gold_agg_constructor_season' (40 linhas) com a suite 'gold_agg_constructor_season'
22:08:30 | INFO    | f1_pipeline.quality.context  | APROVADO | gold_agg_constructor_season: 3/3 regras aprovadas


,tabela,regras,aprovadas,reprovadas,taxa_sucesso_%,linhas,status
0,gold_agg_driver_season,7,7,0,100.0,89,APROVADO
1,gold_agg_constructor_season,3,3,0,100.0,40,APROVADO


In [22]:
# Reconciliação manual: a soma da gold bate com a soma da silver?
pontos_silver = silver["fact_result"].groupby("season")["points"].sum()
pontos_gold = gold["agg_driver_season"].groupby("season")["pontos"].sum()

conferencia = pd.DataFrame({"silver": pontos_silver, "gold": pontos_gold})
conferencia["diferença"] = conferencia["gold"] - conferencia["silver"]
conferencia


,silver,gold,diferença
season,,,
2021,2189.5,2189.5,0.0
2022,2242.0,2242.0,0.0
2023,2242.0,2242.0,0.0
2024,2443.0,2443.0,0.0


> **Por que conferir isso na mão também?** Porque a expectativa do GX testa
> *faixas e relações*; esta conferência testa *identidade*. As duas juntas cobrem
> muito mais do que cada uma sozinha.
>
> **Nota sobre os totais:** eles ficam abaixo do campeonato oficial porque este
> projeto não ingere as corridas **sprint** (endpoint `/{season}/sprint`).
> É proposital — vira o exercício 1 do notebook 01.

In [23]:
from f1_pipeline.quality import publish_data_docs, save_quality_report

save_quality_report(resultados_silver + resultados_gold, etapa="processamento")
publish_data_docs(contexto)

from f1_pipeline.utils.io import describe_layer, human_size

for camada in (config.BRONZE_DIR, config.SILVER_DIR, config.GOLD_DIR):
    inventario = describe_layer(camada)
    print(f"{camada.name:<8} {len(inventario):>2} arquivos  {human_size(inventario['bytes'].sum()):>9}")


22:08:37 | INFO    | f1_pipeline.quality.context  | Relatório de qualidade salvo em C:\dev\Estudo-MDS\reports\quality_report.json
22:08:39 | INFO    | f1_pipeline.quality.context  | Data Docs publicados em C:\dev\Estudo-MDS\docs\data_docs\index.html
bronze    6 arquivos   196.4 KB
silver    9 arquivos   164.5 KB
gold      7 arquivos    80.0 KB


## 7. O que fica desta fase

* **Tipar não é burocracia**: é o que torna o dado somável, ordenável e
  comparável.
* **A regra de negócio mora na silver**, uma vez só, para o projeto inteiro.
* **Defina o grão antes de agregar.** "Uma linha representa o quê?" é a pergunta
  que evita contagem dupla.
* **Modelo estrela**: dimensões descrevem, fatos medem.
* **Outlier não é lixo.** Marque, documente e deixe quem consome decidir.
* **A gold é modelada pela pergunta**, não pelo dado. Se o consumidor precisa de
  JOIN, a gold está incompleta.
* **Reconcilie.** Agregação que não bate com o detalhe é bug, não arredondamento.

### Exercícios

1. Crie `agg_pit_stop_por_equipe`: qual equipe tem o pit stop mais rápido em cada
   temporada? (lembre-se de excluir as paradas atípicas)
2. Acrescente à `agg_driver_season` a coluna `pontos_por_volta_liderada`... e
   descubra por que ela é impossível com os dados que ingerimos. Que endpoint
   faltaria?
3. A regra do duelo de corrida exige que os dois pilotos terminem. Implemente uma
   variante que conte também os GPs em que apenas um abandonou, e compare os dois
   rankings de 2024. Qual conta melhor a história?

---

**Próximo:** [`04_analytics.ipynb`](04_analytics.ipynb) — colocar isso na mão de
quem decide.